In [3]:
!pip install triton

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.1 MB/s eta 0:00:00


In [3]:
!pip install git+https://github.com/huggingface/transformers

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-0fob7o6_
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-0fob7o6_
  Resolved https://github.com/huggingface/transformers to commit 349e00c1a367ce263624e525038250625dcf20c7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.2.0.dev0-py3-none-any.whl size=11461965 sha256=d50e3f85d8cb6c3e34033c9dcb40bd171e07979eb280dfe919ba288c58e6f0b3
  Stored in directory: /tmp/pip-ephem-wheel-cache-x8ad3u5o/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
!pip install mistral-common --upgrade

  Using cached mistral_common-1.9.1-py3-none-any.whl.metadata (5.3 kB)
  Using cached pydantic_extra_types-2.11.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached pycountry-24.6.1-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 79.2 MB/s eta 0:00:00


In [1]:
# scripts/analyze_mistral_architecture.py
import torch
from transformers import AutoTokenizer, Mistral3ForConditionalGeneration

MODEL = "mistralai/Ministral-3-3B-Instruct-2512"

def analyze_mistral_architecture():
    print("=" * 80)
    print(f"Analyzing {MODEL} architecture")
    print("=" * 80)

    # Load model and tokenizer
    print("\n1. Loading model...")
    tok = AutoTokenizer.from_pretrained(MODEL)
    model = Mistral3ForConditionalGeneration.from_pretrained(
        MODEL,
        torch_dtype=torch.float16,
        device_map="cpu",
    )

    print(f"\n2. Model type: {type(model)}")

    # Top-level structure
    print("\n3. Top-level modules:")
    for name, module in model.named_children():
        print(f"   - {name}: {type(module).__name__}")

    # Model structure
    print("\n4. Model structure (model submodule):")
    if hasattr(model, "model"):
        print(f"   model type: {type(model.model).__name__}")
        for name, module in model.model.named_children():
            print(f"   - model.{name}: {type(module).__name__}")

    # Look for layers
    print("\n5. Looking for transformer layers...")
    layer_container = None
    layer_path = None

    if hasattr(model, "model"):
        if hasattr(model.model, "layers"):
            layer_container = model.model.layers
            layer_path = "model.model.layers"
        elif hasattr(model.model, "transformer") and hasattr(model.model.transformer, "h"):
            layer_container = model.model.transformer.h
            layer_path = "model.model.transformer.h"
        elif hasattr(model.model, "encoder") and hasattr(model.model.encoder, "layer"):
            layer_container = model.model.encoder.layer
            layer_path = "model.model.encoder.layer"
        elif hasattr(model.model, "decoder") and hasattr(model.model.decoder, "layers"):
            layer_container = model.model.decoder.layers
            layer_path = "model.model.decoder.layers"

    if layer_container is not None:
        print(f"   ✓ Found layers at: {layer_path}")
        print(f"   Number of layers: {len(layer_container)}")

        # Analyze first layer structure
        print(f"\n6. First layer structure ({layer_path}[0]):")
        first_layer = layer_container[0]
        print(f"   Layer type: {type(first_layer).__name__}")

        # List all submodules in the first layer
        print("\n   Submodules in first layer:")
        for name, module in first_layer.named_children():
            print(f"   - {name}: {type(module).__name__}")

            # If it's an attention module, explore its structure
            if 'attn' in name.lower():
                print(f"     Attention submodules:")
                for subname, submodule in module.named_children():
                    print(f"       - {subname}: {type(submodule).__name__}")

            # If it's an MLP/FFN module, explore its structure
            if any(x in name.lower() for x in ['mlp', 'ffn', 'feed']):
                print(f"     MLP submodules:")
                for subname, submodule in module.named_children():
                    print(f"       - {subname}: {type(submodule).__name__}")
    else:
        print("   ✗ Could not find layers in expected locations")

        # Try to find any sequence of layers
        print("\n   Searching for layer-like structures...")
        def find_layers(module, path="", depth=0, max_depth=3):
            if depth > max_depth:
                return
            for name, child in module.named_children():
                current_path = f"{path}.{name}" if path else name
                if hasattr(child, '__len__') and not isinstance(child, torch.nn.Module):
                    if len(child) > 0 and hasattr(child[0], 'named_children'):
                        print(f"   Possible layer container at: {current_path} (len={len(child)})")
                find_layers(child, current_path, depth + 1, max_depth)

        find_layers(model)

    # Test forward pass with a tiny input to see hidden states
    print("\n7. Testing forward pass to understand output structure...")
    test_input = tok("Hello", return_tensors="pt")
    with torch.no_grad():
        outputs = model(**test_input, output_hidden_states=True)

    print(f"   output_hidden_states available: {hasattr(outputs, 'hidden_states')}")
    if hasattr(outputs, 'hidden_states'):
        print(f"   Number of hidden states: {len(outputs.hidden_states)}")
        print(f"   Hidden state shapes:")
        for i, hs in enumerate(outputs.hidden_states):
            print(f"   - hs[{i}]: {hs.shape}")

    # Check for lm_head
    print("\n8. LM Head:")
    if hasattr(model, 'lm_head') or hasattr(model, 'embed_out'):
        head_name = 'lm_head' if hasattr(model, 'lm_head') else 'embed_out'
        head = getattr(model, head_name)
        print(f"   Found {head_name}: {type(head).__name__}")
        if hasattr(head, 'weight'):
            print(f"   Weight shape: {head.weight.shape}")

    return model, layer_path, layer_container

if __name__ == "__main__":
    model, layer_path, layers = analyze_mistral_architecture()

Analyzing mistralai/Ministral-3-3B-Instruct-2512 architecture

1. Loading model...


Using FP8 quantized models requires a GPU or XPU, we will default to dequantizing the model to bf16 since no GPU or XPU is available


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]


2. Model type: <class 'transformers.models.mistral3.modeling_mistral3.Mistral3ForConditionalGeneration'>

3. Top-level modules:
   - model: Mistral3Model
   - lm_head: Linear

4. Model structure (model submodule):
   model type: Mistral3Model
   - model.vision_tower: PixtralVisionModel
   - model.multi_modal_projector: Mistral3MultiModalProjector
   - model.language_model: Ministral3Model

5. Looking for transformer layers...
   ✗ Could not find layers in expected locations

   Searching for layer-like structures...

7. Testing forward pass to understand output structure...
   output_hidden_states available: True
   Number of hidden states: 27
   Hidden state shapes:
   - hs[0]: torch.Size([1, 1, 3072])
   - hs[1]: torch.Size([1, 1, 3072])
   - hs[2]: torch.Size([1, 1, 3072])
   - hs[3]: torch.Size([1, 1, 3072])
   - hs[4]: torch.Size([1, 1, 3072])
   - hs[5]: torch.Size([1, 1, 3072])
   - hs[6]: torch.Size([1, 1, 3072])
   - hs[7]: torch.Size([1, 1, 3072])
   - hs[8]: torch.Size([1, 